- Time 관련 변수 보강 후: amyloid beta diagnosis

***

# Data Merge

### KR Font

In [13]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스(-) 기호 깨지는 것 방지

In [1]:
import sys
sys.path.append("..")  # 상위 폴더를 파이썬이 찾는 경로에 추가

from raw_to_merge_enhanced import preprocess_and_merge

### Dataset이 다름, 455 version만 소검사 모두 존재. $\to$ 따라서 889 version은 사용 불가. (소검사 RT 등 사용하는 경우)

In [2]:
raw_path = "../../Data/Raw/scstBiomark_vaccine_naive_455_deidentified.xlsx"
save_path = "../../Data/Preprocessed/scstBiomark_vaccine_naive_455_deidentified_merged_enhanced.xlsx"

In [3]:
preprocess_and_merge(raw_path = raw_path, save_path = save_path)

=== Enhanced features 생성 ===
  [visual_forward] pattern='(?i)^VST_forward_RT_.*_TotalTime$' -> 14개 매칭: ['VST_forward_RT_2-1st_TotalTime', 'VST_forward_RT_2-2nd_TotalTime', 'VST_forward_RT_3-1st_TotalTime', 'VST_forward_RT_3-2nd_TotalTime', 'VST_forward_RT_4-1st_TotalTime', 'VST_forward_RT_4-2nd_TotalTime', 'VST_forward_RT_5-1st_TotalTime', 'VST_forward_RT_5-2nd_TotalTime', 'VST_forward_RT_6-1st_TotalTime', 'VST_forward_RT_6-2nd_TotalTime', 'VST_forward_RT_7-1st_TotalTime', 'VST_forward_RT_7-2nd_TotalTime', 'VST_forward_RT_8-1st_TotalTime', 'VST_forward_RT_8-2nd_TotalTime']
  [visual_backward] pattern='(?i)^VST_backward_RT_.*_TotalTime$' -> 14개 매칭: ['VST_backward_RT_2-1st_TotalTime', 'VST_backward_RT_2-2nd_TotalTime', 'VST_backward_RT_3-1st_TotalTime', 'VST_backward_RT_3-2nd_TotalTime', 'VST_backward_RT_4-1st_TotalTime', 'VST_backward_RT_4-2nd_TotalTime', 'VST_backward_RT_5-1st_TotalTime', 'VST_backward_RT_5-2nd_TotalTime', 'VST_backward_RT_6-1st_TotalTime', 'VST_backward_RT_6-2nd_Total

## Data Load

In [9]:
import numpy as np
import pandas as pd
path = save_path
df = pd.read_excel(path, sheet_name='all')

In [11]:
df

,user_ID,1st_amyloid_status,1st_cognitive_status,APOE,sex,education_category,졸업여부,education_year,AGE,visual_forward_Correct_z_score,...,immediate_free_recall_trial_1_RT_raw_score,immediate_free_recall_trial_2_RT_raw_score,immediate_free_recall_trial_3_RT_raw_score,delayed_free_recall_RT_raw_score,word_recognition_RT_raw_score,place_recognition_RT_raw_score,trailmaking_part_a_success_ratio,trailmaking_part_b_RT_digit_to_days_of_the_week,trailmaking_part_b_RT_days_of_the_week_to_digit,trailmaking_part_b_success_ratio
0,183,음성,SCD,E3/E4,여성,고등학교,졸업,12,69,0.02,...,46835.0,27452.0,17850,37780,20820,19316,1.0,9930,19978,1.0
1,248,음성,MCI,E3/E4,여성,대학원(석사),졸업,18,81,-0.26,...,60154.0,60145.0,33979,34300,34486,31918,1.0,30984,26407,1.0
2,265,양성,Dementia,E3/E4,여성,대학교,졸업,16,87,0.06,...,60151.0,47840.0,40283,15349,53076,65702,1.0,96537,34336,1.0
3,283,양성,MCI,E3/E4,여성,대학교,졸업,16,73,1.07,...,53489.0,41649.0,60155,60157,32111,20969,1.0,11550,12385,1.0
4,426,음성,SCD,E3/E3,여성,초등학교,중퇴,4,84,0.97,...,58514.0,60161.0,48689,58299,41952,78138,1.0,17509,55356,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,9570,음성,MCI,E3/E3,여성,대학교,졸업,16,72,0.29,...,0.0,0.0,0,0,26699,33371,1.0,19711,20133,1.0
421,9582,양성,Dementia,E3/E3,남성,고등학교,졸업,12,66,0.28,...,0.0,0.0,0,0,39096,39647,1.0,24226,36014,1.0
422,9614,음성,MCI,E2/E3,여성,대학교,졸업,14,77,0.18,...,NaN,NaN,0,0,36232,31560,1.0,38957,80006,1.0
423,9615,음성,MCI,E3/E3,여성,대학교,졸업,16,82,-0.12,...,0.0,0.0,0,0,47373,59510,1.0,65055,47030,1.0


- 분포 체크

In [10]:
pd.crosstab(
    df['1st_cognitive_status'].astype(pd.CategoricalDtype(['SCD', 'MCI', 'Dementia'], ordered=True)),
    df['1st_amyloid_status'],
    margins=True, margins_name='Total'
)

1st_amyloid_status,양성,음성,Total
1st_cognitive_status,,,
SCD,20,109,129
MCI,85,136,221
Dementia,42,33,75
Total,147,278,425


- 예전 분석 `Project/03. vaccine_naive_889_dataset/01_feature_selection_v1.ipynb` 에서 features가 추가되었으므로, 현 상태론 모델링 의미가 없음.
- 따라서 여기서 `RT` 등 시간 관련 추가 변수를 추가하는 작업은 중단. 성능이 더욱 박살날 것임.
- 결국, dataset 사이즈 문제로 `N=889` 버전 + 기존 composite scores만으로 모델링을 마침.

### DROP columns

In [12]:
df = df.drop(columns=["education_category", "졸업여부"], errors="ignore")

### Fill NA

In [13]:
# df 전체 컬럼에서 NaN 개수 확인
na_counts = df.isna().sum()
print(na_counts[na_counts > 0])

immediate_free_recall_trial_1_RT_raw_score    1
immediate_free_recall_trial_2_RT_raw_score    1
dtype: int64


In [14]:
na_fill_cols = ['immediate_free_recall_trial_1_RT_raw_score', 'immediate_free_recall_trial_2_RT_raw_score']

for col in na_fill_cols:
    n_na = df[col].isna().sum()
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)
    print(f"{col}: NaN {n_na}개 -> 평균값 {mean_val:.4f}로 채움")

immediate_free_recall_trial_1_RT_raw_score: NaN 1개 -> 평균값 30361.9363로 채움
immediate_free_recall_trial_2_RT_raw_score: NaN 1개 -> 평균값 30272.1580로 채움


***

## Experiments